# Avaliacao do Modelo - CreditGuard AI

Carrega o modelo salvo em Model/artifacts/ e avalia seu desempenho no conjunto de teste.
O split e recriado deterministicamente com random_state=42 a partir do abt.csv.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

In [ ]:
with open("Model/config.yaml", "r") as f:
    model_cfg = yaml.safe_load(f)

with open("DataPipeline/config.yaml", "r") as f:
    pipeline_cfg = yaml.safe_load(f)

model    = joblib.load(model_cfg["artifacts"]["model"])
features = joblib.load(model_cfg["artifacts"]["features"])

print(f"Modelo carregado: {model_cfg[chr(39)]artifacts[chr(39)][chr(39)]model[chr(39)]}")
print(f"Features: {len(features)} colunas")

In [ ]:
df = pd.read_csv(pipeline_cfg["paths"]["abt"])

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

_, X_test, _, y_test = train_test_split(
    X, y,
    test_size=model_cfg["split"]["test_size"],
    random_state=model_cfg["split"]["random_state"],
    stratify=y
)

print(f"Conjunto de teste: {X_test.shape}")

## Metricas de Avaliacao

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall":    recall_score(y_test, y_pred),
    "F1-Score":  f1_score(y_test, y_pred),
    "ROC-AUC":   roc_auc_score(y_test, y_proba),
}

print("Modelo: XGBoost Balanced (candidato a producao)
")
for k, v in metrics.items():
    print(f"  {k:12}: {v:.4f}")

## Matriz de Confusao

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Adimplente", "Inadimplente"],
    yticklabels=["Adimplente", "Inadimplente"]
)
plt.title("Matriz de Confusao - XGBoost Balanced")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

## Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"XGBoost Balanced (AUC = {metrics[chr(39)]ROC-AUC[chr(39)]:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Aleatorio (AUC = 0.500)")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")
plt.title("Curva ROC - CreditGuard AI")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusao

O modelo XGBoost Balanced e o candidato atual para producao.
Proximo passo: comparar com Random Forest e LightGBM antes da promocao definitiva.